# The Scaling Imperative

## TLDR

Foundation models keep getting better as they get bigger and see more data. That
is a measured power law. But a bigger model and a bigger batch
quickly outgrow a single GPU, both in raw compute and in memory. This notebook
explains why we scale, where the memory actually goes, and the handful of ways
to split a training job across many GPUs. The rest of the course turns each of
those ideas into running Ray code on four GPUs.

## Introduction

In 2012 AlexNet was trained on two GPUs and kicked off the deep learning era.
Training a neural network is, at its core, a pile of matrix multiplications, and
GPUs do those far faster than CPUs. A decade of scaling laws later, the recipe
for a more capable model is simple to state and expensive to run. Use more
parameters, feed more tokens, and you get a lower loss.

The catch is that a modern foundation model does not fit on one GPU. A 1 billion
parameter model needs roughly 28 GB just to train, and a 70 billion parameter
model needs around 2 TB. No single accelerator has that. So we spread the work
across many GPUs, and the interesting engineering is in how we split it and how
we keep the GPUs busy while they talk to each other.

This is where Ray comes in. Ray turns a cluster of CPUs and GPUs into one system
you program with a few primitives, and Anyscale runs that cluster for you. By
the end of the course you will have used the same small set of Ray calls to go
from one GPU to many, to shard a model that does not fit, and to recover from a
worker that dies mid run.


## Key concepts used in this notebook

**Scaling laws.** Loss falls as a power law in model size and dataset size
(Kaplan et al. 2020, Chinchilla / Hoffmann et al. 2022). Bigger plus more data
means better, within a compute budget.

**The memory wall.** Training memory is parameters plus gradients plus optimizer
states plus activations. For Adam in mixed precision this is far more than the
model weights alone.

**The five parallelism axes.** Data, tensor, pipeline, sequence or context, and
expert parallelism. Each one splits a different part of the problem and pays a
different communication cost.

**Sharding.** Instead of replicating the model on every GPU, partition the
parameters, gradients, and optimizer states across GPUs. This is what DeepSpeed
ZeRO and PyTorch FSDP do.

**The bandwidth hierarchy.** On-chip cache is faster than GPU memory, which is
faster than NVLink, which is faster than the network, which is faster than PCIe.
The right parallelism strategy is the one that matches your bandwidth hierarchy.


## What you will learn

- Why bigger models and more data reliably improve foundation models
- Why a GPU, not a CPU, is the unit of training compute
- Where training memory actually goes, and how to estimate it
- The two families of memory reduction, recompute and offload
- The five ways to split a training job across GPUs, at a glance
- Why network and interconnect bandwidth, not FLOPs, is often the real limit

## Why Ray on Anyscale for foundation model training

| Challenge | Without Ray | With Ray on Anyscale |
|---|---|---|
| Launch many GPU workers | Manual SSH, process groups, env wiring on each node | One `TorchTrainer` plus a `ScalingConfig` |
| Mixed CPU and GPU cluster | Hand-managed placement | Ray schedules and autoscales the cluster |
| Move from 4 to 400 GPUs | Rewrite the launch and data plumbing | Change one number in `ScalingConfig` |
| One node or multiple | Different launch scripts | Same code, Ray places the workers |
| Shared storage for checkpoints | Configure NFS or object store by hand | `/mnt/cluster_storage` is there on every node |

The theory in this notebook is framework agnostic. The point of the rest of the
course is that Ray is how you actually run it, and Anyscale is how you run it at
scale without managing the cluster yourself.


## Why train on GPUs

Training is matrix multiplication, and that is exactly what GPUs are built for.
A single AlexNet style multiply of a 256 by 4096 input against a 4096 by 4096
weight matrix is about 8.6 GFLOP. The hardware gap is enormous and it has only
widened.

| Hardware | Year | Peak throughput | Precision |
|---|---|---|---|
| Intel i7-3960X (CPU) | 2011 | 0.1 TFLOPS | FP32 |
| NVIDIA GTX 580 (GPU) | 2011 | 1.58 TFLOPS | FP32 |
| NVIDIA H100 (GPU) | 2022 | 989 TFLOPS | BF16 |

GPU throughput grew roughly 1000 times from 2011 to 2022. Training a deep
network is compute intensive, and GPUs win because they do parallel matrix
operations. That is the entire reason this is a GPU course.


## Why bigger and more data wins

Two observations, both formalized as power laws, drive the whole field.

**More data lowers loss.** Train the same model on more tokens and the loss
keeps falling along a smooth curve. Llama 2 7B trained on 2 trillion tokens is
the canonical public example.

**Bigger models lower loss.** Hold the data fixed and grow the parameter count,
and the loss falls again. Llama 2 shipped at 7B, 13B, 34B, and 70B, and each
larger model reaches a lower loss on the same data.

Put together, better performance comes from larger models plus more data, spent
within a compute budget (Chinchilla). The trend is visible across a decade of
flagship models.

| Model | Year | Parameters | Training tokens |
|---|---|---|---|
| GPT-3 | 2020 | 175B | 300B |
| Llama 2 | 2023 | 70B | 2T |

The takeaway is that scale is not optional if you want a competitive model, and
scale means distributed training.


## The memory wall

The reason one GPU is not enough is memory, not just speed. Training memory has
four parts. Take a 1 billion parameter model trained with Adam in mixed
precision.

| Component | How to size it | 1B model |
|---|---|---|
| Parameters (fp16) | params x 2 bytes | 2 GB |
| Gradients (fp16) | params x 2 bytes | 2 GB |
| Optimizer states (Adam, fp32) | params x 2 moments x 4 bytes | 8 GB |
| Activations | batch x layers x hidden x bytes | ~16 GB |
| **Total** | | **~28 GB** |

Two things jump out. The Adam optimizer states are four times the size of the
model itself, because Adam keeps two fp32 moments per parameter. And activations
dominate at training time, because you must keep them for the backward pass.

Now scale that up. A 70 billion parameter model needs roughly 2 TB of training
memory. A single T4 has 16 GB and even an H100 has 80 GB. The model does not
fit, so we need strategies to reduce memory and strategies to split the model
across GPUs.


## Reducing memory before splitting

Before splitting the model, there are two ways to shrink its footprint on each
GPU.

**Activation checkpointing.** Do not store every layer's activations. Keep a few
checkpoints and recompute the rest during the backward pass. This trades compute
for memory, roughly a 30 percent compute increase for up to a 4 times reduction
in activation memory. In practice memory is more precious than compute, so this
can actually be worth it.

**Offloading.** Push data you do not need right now to a slower but larger tier,
typically CPU memory. Optimizer states, inactive parameters, and activations can
all live on the CPU and come back when needed. The cost is bandwidth. Fetching
from CPU over PCIe is about 50 times slower than reading from GPU memory, so
offloading only pays off when the data is accessed infrequently.

Both of these buy headroom on a single GPU. To go further, you split the work
across GPUs, and that means picking a parallelism strategy.


## The five ways to split work

There are five common axes along which a foundation model training job can be parallelized. Real runs often
combine several of them.

| Axis | What it splits | What it communicates |
|---|---|---|
| Data parallel (DP) | The batch across model replicas | All-reduce of gradients |
| Tensor parallel (TP) | Weight matrices within a layer | All-reduce of activations each layer |
| Pipeline parallel (PP) | Layers into sequential stages | Activations handed between stages |
| Sequence or context (SP, CP) | The sequence dimension | All-to-all for long-context attention |
| Expert parallel (EP) | The experts of a mixture-of-experts | All-to-all routing of tokens |

Cutting across all of these is **sharding**, the idea behind DeepSpeed ZeRO and
PyTorch FSDP. Rather than replicate the full model on every data-parallel GPU,
partition the parameters, gradients, and optimizer states across them and gather
each piece only when it is needed. Sharding is how data parallelism stops
wasting memory on redundant copies.

This course goes deep on the axes you can actually run on four GPUs, which are
data and tensor parallelism plus sharding. Pipeline and expert parallelism are
covered as concepts, because they need far more hardware to be meaningful.


## Bandwidth is the real bottleneck

Splitting work means GPUs must talk to each other, and communication runs at
very different speeds depending on where it happens.

| Tier | Bandwidth | Scope |
|---|---|---|
| Cache | ~20 TB/s | On-chip |
| HBM (GPU memory) | ~3.35 TB/s | On-GPU |
| NVLink | ~900 GB/s | Within a node |
| InfiniBand | ~200 GB/s per link | Across nodes |
| PCIe 5.0 | ~64 GB/s | Host to device |

The guiding principle for the whole course is to match the parallelism strategy
to this hierarchy. Put chatty, latency-sensitive communication like tensor
parallelism on the fastest link you have, and put communication that can overlap
with compute, like data-parallel gradient sync, across the slower network.

## How this maps to the rest of the course

| This notebook taught | Notebook that makes it real |
|---|---|
| Why we scale, the memory wall | 00 (here) |
| One GPU to many with little code change | 01 Ray foundations |
| Activation checkpointing, sharding (FSDP, ZeRO) | 02 Memory and sharding |
| Tensor plus data parallelism, the bandwidth tradeoff | 03 2D parallelism |
| Recovering from failure, seeing what the GPUs do | 04 Fault tolerance and observability |
| Combining all five axes | 05 Putting it together |

Now let us confirm the cluster we are working on, then go build.


## Cell 1 — Confirm the cluster

**What you do.** Connect to the running Ray cluster and print its resources.

**What to check.** You should see 4 GPUs and a Tesla T4 accelerator type. The
head node is CPU only, so the GPUs live on the worker node.

**Why it matters.** Every later notebook runs on exactly this cluster. The
standard runtime environment from `common.utils` ships our shared code to every
worker and sets the env vars these T4s need.


In [ ]:
import os
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"  # quiet a harmless driver tip

import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

resources = utils.print_cluster_resources()

## Cell 2 — See the GPU precision reality

**What you do.** Run a tiny task on one GPU worker that reports what the T4
claims about bfloat16 versus what it can do natively.

**What to check.** `is_bf16_supported()` returns True, but native bf16 is False.
The dtype we will train in comes back as float16.

**Why it matters.** The naive PyTorch check counts slow software emulation, so
it would steer you to bf16 on a GPU that has no native bf16. We pick precision
by native support instead, which is why every mixed-precision path in this
course uses fp16 on the T4. This is a real lesson, not a workaround.


In [ ]:
@ray.remote(num_gpus=1)
def probe_precision():
    import torch
    from common import utils
    return {
        "gpu": torch.cuda.get_device_name(0),
        "is_bf16_supported (naive)": torch.cuda.is_bf16_supported(),
        "native bf16": utils.has_native_bf16(),
        "dtype we will train in": str(utils.mixed_precision_dtype()),
    }

ray.get(probe_precision.remote())

## Conclusion

Scale is the price of a competitive foundation model. Power laws reward more
parameters and more data, GPUs are the only hardware that makes the matrix math
affordable, and a real model overflows a single GPU on memory long before it
overflows on time. You reduce memory with activation checkpointing and
offloading, then you split the job along one or more of five axes, always
watching the bandwidth hierarchy.

Ray primitives you met here. `ray.init`, `ray.cluster_resources`, and a remote
task with `@ray.remote(num_gpus=1)`.

Next, in notebook 01, you take a single-GPU PyTorch loop and turn it into a
distributed four-GPU job with `prepare_model`, Ray Data, and a `TorchTrainer`,
changing almost nothing about the loop itself.
